# Part 4 — Building with Work IQ tools + actions

Reading context is half the story. To **change something** in Microsoft 365 — send mail, create an
event, reply to a thread — you use **`do_action`**, Work IQ's **only write path**.

`do_action` runs as the signed-in user, so it can only do what that user is allowed to do, and it
respects sensitivity labels and compliance policy. Always confirm with the user before writing.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, call_tool

token = get_user_token()
def mcp_call(name, arguments):
    return call_tool(token, name, arguments)
print("Signed in - Work IQ action helper ready. Token:", token[:16], "...")

## 1. Discover the action schema

Use `get_schema` on the action path so the model knows the exact payload.

In [ ]:
# Discover the request body for an action (operationType="action").
schema = mcp_call("get_schema", {"path": "/me/sendMail", "operationType": "action"})
print(schema["result"]["content"][0]["text"][:1000])

## 2. Draft the follow-up (read first)

Ground the action in real context — e.g. summarize the manager's latest thread before replying.

In [ ]:
draft = mcp_call("ask", {"question":
    "Draft a short, friendly follow-up reply to my manager's most recent email. Return only the body."})
print(draft["result"]["structuredContent"]["answer"])

## 3. Send it with `do_action`

> Only run this cell after a human has approved the draft.

In [ ]:
# do_action performs the write: actionUrl + jsonBody (shape comes from get_schema above).
SEND = False  # flip to True to actually send
action_url = "/me/sendMail"
json_body = {
    "Message": {
        "subject": "Re: your note",
        "body": {"contentType": "Text", "content": "<approved draft here>"},
        "toRecipients": [{"emailAddress": {"address": "manager@contoso.com"}}],
    },
    "SaveToSentItems": True,
}
if SEND:
    print(mcp_call("do_action", {"actionUrl": action_url, "jsonBody": json_body}))
else:
    print("Dry run - set SEND=True to send. do_action args:")
    print(json.dumps({"actionUrl": action_url, "jsonBody": json_body}, indent=2))

### Recap

- `do_action` is the single write verb; everything else reads or discovers.
- Pattern: **read context → draft → confirm → `do_action`**.

**Next:** Part 5 — wrap this into a Microsoft Agent Framework agent.